# Hive DDL и таблицы — 30 заданий

Практика выполняется на eBay в `/data/raw/ebay`. Решений нет.

## Результаты обучения

После **Hive DDL** вы должны объяснить внутренний механизм, предсказать изменения metadata/files, выбрать безопасную команду и доказать итог измерением, а не сообщением об успехе.

## Архитектурная модель

Hive Metastore хранит schema, LOCATION и partitions, а строки лежат в HDFS. Managed/external различаются владением lifecycle; schema-on-read не исправляет несовместимые типы.

```text
client ── metadata RPC ──► NameNode
  │                         │ block locations
  └── data stream ──► DataNode 1 ──► DataNode 2

HiveServer2 ──► Metastore (schema/location/partitions)
      └──────► execution engine ──► HDFS files
```
NameNode не хранит содержимое файла, а Metastore не хранит строки таблицы.

## Физическая схема eBay

```text
/data/raw/ebay/
├── snapshot_dt=2026-06-24/part-....snappy.parquet
├── snapshot_dt=2026-06-25/part-....snappy.parquet
└── ...
```
Grain: наблюдение `itemid` в `snapshot_dt`. Группы колонок: карточка/цена,
иерархия категорий, продавец, география и доставка. Полная schema — в `data-catalog`.

Общий raw read-only; результаты принадлежат `/user/$HDFS_USER/hadoop_training` и личной Hive DB.

## Алгоритм исследования

1. Зафиксируйте path/URI, owner и ожидаемый объект. 2. Снимите состояние до. 3. Выполните одно изменение. 4. Проверьте exit code. 5. Измерьте namespace/files/bytes/schema/rows. 6. Повторите команду и оцените идемпотентность. 7. Сохраните evidence.

Разделяйте metadata operation и file operation, подтверждайте SHOW CREATE/DESCRIBE и HDFS ls/count.

## Типичные ошибки

- Путать локальный путь с HDFS URI.
- Делать вывод по `ls`, не проверяя blocks/bytes/schema.
- Использовать root или 777 вместо модели доступа.
- Создавать partition-каталог без Metastore или metadata без файлов.
- Считать replication резервной копией.
- Игнорировать малые файлы и цену NameNode metadata.

## Самопроверка

1. Какие metadata изменятся? 2. Где физически лежат bytes? 3. Сколько logical и physical bytes? 4. Кто может читать/писать? 5. Что произойдёт при повторе? 6. Какая независимая команда опровергнет вывод?

## Ментальная модель

Hive-таблица — metadata над файлами. EXTERNAL отделяет жизненный цикл данных от каталога, managed связывает их. LOCATION, схема и физический Parquet должны согласовываться.

## Подробная теория

### 1. Metastore

Hive хранит DDL, schema, location и partitions в metastore; строки остаются в HDFS. DROP затрагивает данные по-разному для managed и external.

### 2. Схема при чтении

Schema-on-read не исправляет плохие данные: несовпадение физического Parquet-типа и DDL приводит к NULL или ошибке.

### 3. LOCATION

Путь — часть контракта. Две таблицы могут смотреть на одни файлы, поэтому владение и жизненный цикл нужно определить заранее.

### 4. Запись

INSERT INTO добавляет, INSERT OVERWRITE заменяет выбранную область. CTAS удобен, но типы и свойства следует проверить SHOW CREATE TABLE.

### 5. Сложные типы

ARRAY, MAP и STRUCT сохраняют вложенность, однако усложняют совместимость и запросы. Выбор должен следовать форме события.

## Стенд

NameNode `namenode:8020`, два DataNode, HiveServer2 `hiveserver2:10000`. Личные артефакты не создаются в общем read-only raw-слое.

## Как сдаётся задание

Валидатор проверяет артефакт и JSON-доказательство. В `command` запишите фактическую команду, в `observation` — измеренный результат, в `explanation` — почему он получился. Минимальная длина защищает от пустых ответов; содержательный смысл остаётся вашей ответственностью.

In [ ]:
import json, os, subprocess, tempfile

def save_evidence(module, task, command, observation, explanation):
    user=os.environ.get("HDFS_USER", os.environ.get("HADOOP_USER_NAME", "student"))
    target=f"/user/{user}/hadoop_training/evidence/{module}/task_{task:02d}.json"
    payload={"module":module,"task":task,"command":command,"observation":observation,"explanation":explanation}
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",delete=False,suffix=".json") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2); local=f.name
    subprocess.run(["hdfs","dfs","-mkdir","-p",target.rsplit("/",1)[0]],check=True)
    subprocess.run(["hdfs","dfs","-put","-f",local,target],check=True)
    os.unlink(local)
    print(target)

### Задание 1. database

Создайте Hive-объект `hv_01` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `database`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 1

### Задание 2. external table

Создайте Hive-объект `hv_02` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `external table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 2

### Задание 3. managed table

Создайте Hive-объект `hv_03` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `managed table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 3

### Задание 4. location

Создайте Hive-объект `hv_04` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `location`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 4

### Задание 5. Parquet table

Создайте Hive-объект `hv_05` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `Parquet table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 5

### Задание 6. typed schema

Создайте Hive-объект `hv_06` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `typed schema`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 6

### Задание 7. describe formatted

Создайте Hive-объект `hv_07` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `describe formatted`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 7

### Задание 8. show create

Создайте Hive-объект `hv_08` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `show create`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 8

### Задание 9. load data

Создайте Hive-объект `hv_09` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `load data`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 9

### Задание 10. insert into

Создайте Hive-объект `hv_10` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `insert into`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 10

### Задание 11. insert overwrite

Создайте Hive-объект `hv_11` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `insert overwrite`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 11

### Задание 12. CTAS

Создайте Hive-объект `hv_12` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `CTAS`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 12

### Задание 13. view

Создайте Hive-объект `hv_13` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `view`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 13

### Задание 14. temporary view

Создайте Hive-объект `hv_14` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `temporary view`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 14

### Задание 15. drop external

Создайте Hive-объект `hv_15` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `drop external`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 15

### Задание 16. drop managed

Создайте Hive-объект `hv_16` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `drop managed`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 16

### Задание 17. alter rename

Создайте Hive-объект `hv_17` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `alter rename`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 17

### Задание 18. add columns

Создайте Hive-объект `hv_18` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `add columns`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 18

### Задание 19. table properties

Создайте Hive-объект `hv_19` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `table properties`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 19

### Задание 20. null handling

Создайте Hive-объект `hv_20` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `null handling`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 20

### Задание 21. SerDe

Создайте Hive-объект `hv_21` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `SerDe`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 21

### Задание 22. text table

Создайте Hive-объект `hv_22` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `text table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 22

### Задание 23. repair table

Создайте Hive-объект `hv_23` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `repair table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 23

### Задание 24. metadata vs data

Создайте Hive-объект `hv_24` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `metadata vs data`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 24

### Задание 25. schema mismatch

Создайте Hive-объект `hv_25` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `schema mismatch`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 25

### Задание 26. decimal type

Создайте Hive-объект `hv_26` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `decimal type`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 26

### Задание 27. timestamp type

Создайте Hive-объект `hv_27` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `timestamp type`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 27

### Задание 28. array type

Создайте Hive-объект `hv_28` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `array type`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 28

### Задание 29. map type

Создайте Hive-объект `hv_29` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `map type`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 29

### Задание 30. DDL audit

Создайте Hive-объект `hv_30` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `DDL audit`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py hive_ddl 30

## Итог

Все 30 проверок должны возвращать PASS. Удалять чужие или raw-данные запрещено.